In [2]:
import jax
import jax.numpy as jnp
import jax.random as random
from jax import lax, jit, vmap
from jax.experimental.pjit import pjit
from jax.sharding import PositionalSharding
from functools import partial
import time

# --- Configuration
MAX_RECURSION_DEPTH = 100  # 🚀 Increased to 100 for maximum recursion depth
OPTIMAL_DEPTH_STEP = 50  # 🔥 Increased from 25 to 50 per iteration for max TPU load
DIMENSIONAL_CONSTRAINT = 0.8
BATCH_SIZE = 100_000  # 🚀 Boosted batch size for extreme parallelism

@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / (scale_factor + 1)) * DIMENSIONAL_CONSTRAINT

@jit
def stabilize_depth(depth):
    """Normalizes depth scaling to prevent instability."""
    return depth / (1 + jnp.log1p(depth + 1))

@partial(jit, static_argnames=["depth"])
def dppu_with_dynamic_pi_phi(x, depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    """🔥 Now executes in larger depth chunks to maximize TPU efficiency"""
    depth = stabilize_depth(jnp.minimum(depth, MAX_RECURSION_DEPTH))

    def body_fn(i, val):
        pi_dyn = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT
        new_val = jnp.sin(val * scale * pi_dyn) * jnp.exp(-val / (phi_dyn + 1))
        return new_val

    return lax.fori_loop(0, depth.astype(jnp.int32), body_fn, x)

# --- TPU Sharding Setup
devices = jax.devices()
sharding = PositionalSharding(devices)

batch_input = jnp.linspace(0, 10, BATCH_SIZE)
batch_input = jax.device_put(batch_input, sharding)

batched_dppu_processing = pjit(
    lambda arr: vmap(lambda xi: dppu_with_dynamic_pi_phi(xi, depth=OPTIMAL_DEPTH_STEP, scale_factor=0.5), in_axes=0)(arr),
    in_shardings=(sharding,),
    out_shardings=sharding,
)

# ✅ Max Depth Execution Strategy
def process_with_max_depth(x, total_depth):
    """🔥 Now executing in massive depth chunks"""
    iterations = total_depth // OPTIMAL_DEPTH_STEP
    for _ in range(iterations):
        x = dppu_with_dynamic_pi_phi(x, depth=OPTIMAL_DEPTH_STEP)
    return x

# --- Run Maximum Performance Benchmark
for depth in range(OPTIMAL_DEPTH_STEP, MAX_RECURSION_DEPTH + 1, OPTIMAL_DEPTH_STEP):
    output_batch = process_with_max_depth(batch_input, depth)
    print(f"Batch Output Shape (Depth={depth}):", output_batch.shape)

NUM_TRIALS = 10

# Warm-up compile
_ = dppu_with_dynamic_pi_phi(jnp.ones((BATCH_SIZE,)), depth=OPTIMAL_DEPTH_STEP)

for depth in range(OPTIMAL_DEPTH_STEP, MAX_RECURSION_DEPTH + 1, OPTIMAL_DEPTH_STEP):
    times = []
    for _ in range(NUM_TRIALS):
        start = time.time()
        result = process_with_max_depth(jnp.ones((BATCH_SIZE,)), depth)
        _ = jax.device_get(result)
        end = time.time()
        times.append(end - start)

    avg_time = sum(times) / len(times)
    print(f"\n🔥 TPU Benchmark (Depth={depth}, Batch={BATCH_SIZE})")
    print(f"Avg: {avg_time:.6f}, Min: {min(times):.6f}, Max: {max(times):.6f}")

# --- Investigate TPU Compilation at Extreme Depth
compiled_fn_50 = jax.jit(dppu_with_dynamic_pi_phi).lower(jnp.ones((BATCH_SIZE,)), depth=50)
compiled_fn_100 = jax.jit(dppu_with_dynamic_pi_phi).lower(jnp.ones((BATCH_SIZE,)), depth=100)

print("\n🚀 XLA Compilation for Depth=50:")
print(compiled_fn_50.as_text())

print("\n🚀 XLA Compilation for Depth=100:")
print(compiled_fn_100.as_text())

Batch Output Shape (Depth=50): (100000,)
Batch Output Shape (Depth=100): (100000,)

🔥 TPU Benchmark (Depth=50, Batch=100000)
Avg: 0.000538, Min: 0.000470, Max: 0.000791

🔥 TPU Benchmark (Depth=100, Batch=100000)
Avg: 0.000525, Min: 0.000489, Max: 0.000574


ValueError: Non-hashable static arguments are not supported. An error occurred while trying to hash an object of type <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>, Traced<ShapedArray(int32[], weak_type=True)>with<DynamicJaxprTrace>. The error was:
TypeError: unhashable type: 'DynamicJaxprTracer'
